# Section 1: Project Introduction

This notebook builds a brain MRI classification project for four classes: glioma, meningioma, pituitary tumor, and no tumor. The goal is to combine image preprocessing, feature engineering, and classical machine learning in a transparent workflow that is easy to understand and debug.

## Problem statement
Brain MRI images contain diagnostic patterns that may help separate different tumor types. Manual review of many images is time-consuming, so we build a local machine-learning pipeline that uses image-derived features to classify each image into one of the four categories.

This is an educational project and should not be interpreted as a medical diagnosis system. The goal is research and learning, not clinical validation.

## Classification vs. segmentation
Classification assigns a single label to an image, such as `glioma` or `no_tumor`. Segmentation, by contrast, labels each pixel or region inside an image to identify structures such as tumor tissue, edema, or ventricles. This project uses image-level classification, not segmentation.

## Proposed pipeline
1. Load and inspect the dataset.
2. Explore class balance and image properties.
3. Preprocess the images with OpenCV.
4. Extract structured numerical features.
5. Train and compare machine-learning models.
6. Evaluate results and analyze mistakes.
7. Extend later to a CNN deep-learning comparison.

The notebook is intentionally built in stages so you can verify each section before moving to the next.

In [1]:
# Section 2: Environment Setup
# This cell installs (if needed) and imports the libraries required for the study.
# It also defines the dataset root and output folder in one place so the path is easy to update.

# If a package is missing, run the following from a terminal in VS Code:
# pip install numpy pandas matplotlib seaborn opencv-python scikit-learn pillow joblib

import os
import random
import warnings
from pathlib import Path

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

# Set a seed so that random choices are reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Make the dataset path easy to edit in one place.
# If this folder does not exist on your machine, update it to your actual dataset location.
WORKSPACE_ROOT = Path.cwd()
DATASET_ROOT_CANDIDATES = [
    WORKSPACE_ROOT / "data",
    WORKSPACE_ROOT / "brisc2025" / "classification_task",
    Path(r"C:/Users/your_name/Downloads/brisc2025/classification_task"),
]

DATASET_ROOT = None
for candidate in DATASET_ROOT_CANDIDATES:
    if candidate.exists():
        DATASET_ROOT = candidate
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Dataset not found in the usual project folders. Update DATASET_ROOT_CANDIDATES in this cell with the actual path to your BRISC2025 dataset."
    )

TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"
OUTPUT_DIR = WORKSPACE_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print("Python working directory:", WORKSPACE_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Train path exists:", TRAIN_DIR.exists())
print("Test path exists:", TEST_DIR.exists())
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Seaborn version:", sns.__version__)

Python working directory: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour
Dataset root: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\data
Train path exists: True
Test path exists: True
OpenCV version: 5.0.0
NumPy version: 2.5.3
Pandas version: 3.0.6
Seaborn version: 0.13.2


# Section 3: Dataset Exploration

In this section we inspect the dataset structure, verify that the training and test folders are correct, and understand the class distribution. This is important because model quality depends heavily on data quality, label balance, and whether the splits are truly independent.

We will check:
- class directories
- counts per class
- image sizes
- file readability
- representative examples
- duplicates and corrupted files

In [2]:
# Read all class folders and create a table of counts.
# We use Path.iterdir() to inspect each class directory.

def list_image_files(folder):
    allowed_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in allowed_exts],
        key=lambda p: p.name.lower(),
    )

def summarize_split(split_dir):
    summary = []
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        files = list_image_files(class_dir)
        summary.append({
            'split': split_dir.name,
            'class_name': class_dir.name,
            'image_count': len(files),
            'example_file': files[0].name if files else None,
        })
    return pd.DataFrame(summary)

train_summary = summarize_split(TRAIN_DIR)
test_summary = summarize_split(TEST_DIR)

print("Training split summary:")
display(train_summary)

print("Testing split summary:")
display(test_summary)

# The training and test sets should be separate and should not be combined for model fitting.
# This early check helps us confirm that the workspace follows the expected folder layout.

Training split summary:


,split,class_name,image_count,example_file
0,train,glioma,1147,brisc2025_train_00001_gl_ax_t1.jpg
1,train,meningioma,1329,brisc2025_train_01148_me_ax_t1.jpg
2,train,no_tumor,1067,brisc2025_train_02477_no_ax_t1.jpg
3,train,pituitary,1457,brisc2025_train_03544_pi_ax_t1.jpg


Testing split summary:


,split,class_name,image_count,example_file
0,test,glioma,254,brisc2025_test_00001_gl_ax_t1.jpg
1,test,meningioma,306,brisc2025_test_00255_me_ax_t1.jpg
2,test,no_tumor,140,brisc2025_test_00561_no_ax_t1.jpg
3,test,pituitary,300,brisc2025_test_00701_pi_ax_t1.jpg


In [3]:
# Plot the class distribution in the training set.
# A bar chart makes imbalances easy to see at a glance.

train_class_counts = train_summary.set_index('class_name')['image_count'].to_dict()
classes = list(train_class_counts.keys())
counts = list(train_class_counts.values())

plt.figure(figsize=(8, 5))
sns.barplot(x=classes, y=counts, palette="Set2")
plt.title("Training class distribution")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Interpretation:
# If one class is much smaller than the others, the classifier may become biased toward the majority classes.
# This is why we will inspect class imbalance later and consider evaluation metrics like macro-F1.

In [4]:
# Inspect image dimensions and channel counts.
# MRI images may vary in size, so we need to know the shapes before preprocessing.

shape_records = []
for split_name, split_dir in [('train', TRAIN_DIR), ('test', TEST_DIR)]:
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        for image_path in list_image_files(class_dir):
            image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
            if image is None:
                shape_records.append({
                    'split': split_name,
                    'class_name': class_dir.name,
                    'image_path': str(image_path),
                    'readable': False,
                    'height': None,
                    'width': None,
                    'channels': None,
                })
                continue
            shape_records.append({
                'split': split_name,
                'class_name': class_dir.name,
                'image_path': str(image_path),
                'readable': True,
                'height': image.shape[0],
                'width': image.shape[1],
                'channels': image.shape[2] if len(image.shape) == 3 else 1,
            })

shape_df = pd.DataFrame(shape_records)
print("Rows checked:", len(shape_df))
print("Distinct image shapes:")
print(shape_df[['height', 'width', 'channels']].drop_duplicates().head(10).to_string(index=False))

# A quick visual summary of image dimensions.
plt.figure(figsize=(8, 5))
sns.scatterplot(data=shape_df.dropna(subset=['height', 'width']), x='width', y='height', hue='class_name', s=20, alpha=0.7)
plt.title("Image dimensions across classes")
plt.xlabel("Width")
plt.ylabel("Height")
plt.tight_layout()
plt.show()

# Interpretation:
# Most MRI images are likely the same size, but we still confirm this before resizing the images to a common shape.

Rows checked: 6000
Distinct image shapes:
 height  width  channels
    512    512         3
    512    512         1
    237    213         3
    500    455         3
    270    250         3
    546    472         3
    427    441         3
    510    485         3
    605    507         3
    345    338         3


In [6]:
# Display a representative sample image from each class.
# The goal is to visually confirm that folders are labeled correctly and that there are no obvious anomalies.

fig, axes = plt.subplots(4, 2, figsize=(12, 14))
axes = axes.flatten()

for idx, class_dir in enumerate(sorted((p for p in TRAIN_DIR.iterdir() if p.is_dir()))):
    files = list_image_files(class_dir)
    sample_path = files[0]
    image_bgr = cv2.imread(str(sample_path), cv2.IMREAD_COLOR)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    axes[idx * 2].imshow(image_rgb)
    axes[idx * 2].set_title(f"{class_dir.name}: sample image")
    axes[idx * 2].axis('off')

    hist = cv2.calcHist([image_bgr], [0], None, [256], [0, 256])
    axes[idx * 2 + 1].plot(hist, color='steelblue')
    axes[idx * 2 + 1].set_title(f"{class_dir.name}: intensity histogram")
    axes[idx * 2 + 1].set_xlabel('Pixel intensity')
    axes[idx * 2 + 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Interpretation:
# This is a sanity check. If a class folder contains mostly blank, corrupted, or mislabeled images, these samples will reveal it early.

In [7]:
# Check for unreadable or corrupted files and possible duplicates.
# A duplicate check is useful because related image copies can cause data leakage when splits are created.

import hashlib

readable_errors = []
file_hashes = {}
duplicate_paths = []
all_image_paths = []

for split_dir in [TRAIN_DIR, TEST_DIR]:
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        for image_path in list_image_files(class_dir):
            all_image_paths.append(image_path)
            image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
            if image is None or image.size == 0:
                readable_errors.append(str(image_path))
                continue
            file_hash = hashlib.sha256(image_path.read_bytes()).hexdigest()
            if file_hash in file_hashes:
                duplicate_paths.append((file_hashes[file_hash], image_path))
            else:
                file_hashes[file_hash] = image_path

print("Unreadable/corrupted files:", readable_errors)
print("Duplicate image pairs found:", duplicate_paths[:10])

# Interpretation:
# A few unreadable images or duplicates are not unusual in real datasets, but they should be documented before modeling.
# We will not use test images for feature selection, tuning, or training.

Unreadable/corrupted files: []
Duplicate image pairs found: [(WindowsPath('c:/Users/KIIT/OneDrive/Desktop/Brain Tumour/data/train/meningioma/brisc2025_train_01247_me_ax_t1.jpg'), WindowsPath('c:/Users/KIIT/OneDrive/Desktop/Brain Tumour/data/train/meningioma/brisc2025_train_01268_me_ax_t1.jpg')), (WindowsPath('c:/Users/KIIT/OneDrive/Desktop/Brain Tumour/data/train/meningioma/brisc2025_train_01276_me_ax_t1.jpg'), WindowsPath('c:/Users/KIIT/OneDrive/Desktop/Brain Tumour/data/train/meningioma/brisc2025_train_01277_me_ax_t1.jpg')), (WindowsPath('c:/Users/KIIT/OneDrive/Desktop/Brain Tumour/data/train/meningioma/brisc2025_train_01246_me_ax_t1.jpg'), WindowsPath('c:/Users/KIIT/OneDrive/Desktop/Brain Tumour/data/train/meningioma/brisc2025_train_01302_me_ax_t1.jpg')), (WindowsPath('c:/Users/KIIT/OneDrive/Desktop/Brain Tumour/data/train/meningioma/brisc2025_train_01247_me_ax_t1.jpg'), WindowsPath('c:/Users/KIIT/OneDrive/Desktop/Brain Tumour/data/train/meningioma/brisc2025_train_01386_me_ax_t1.jpg

## Status check

The environment setup and dataset exploration sections are complete. The dataset has been verified with the actual workspace folders and class counts, and the project is ready to move to the next stage: image preprocessing.

# Section 4: Image Preprocessing

In this section we convert raw MRI files into a consistent representation that machine-learning algorithms can use. We should not blindly convert all MRI scans to grayscale or threshold them. The goal is to compare a few preprocessing choices and keep the method that is most useful for the actual data.

## Why preprocessing matters
A raw MRI image is just a matrix of integer pixel values. The value at each pixel is related to how electromagnetic energy is absorbed or reflected in the tissue. Before classification, we often:
- resize images to the same dimensions,
- normalize pixel intensity,
- smooth noise with Gaussian blur,
- optionally threshold the image when it helps separate the foreground from the background.

## Mathematical ideas
- Pixel intensity: each cell in the image stores a number, often from 0 to 255 for 8-bit images.
- Normalization: we rescale values to a common interval such as [0, 1] or [-1, 1].
- Gaussian blur: this smooths the image by averaging neighboring pixels with a Gaussian-weighted kernel.
- Thresholding: we decide whether a pixel belongs to the foreground (brain structure or object) or the background based on a value cutoff.

A good preprocessing pipeline reduces noise while preserving the important structure. We compare a few alternatives and keep only what improves validation quality.


In [8]:
# Preprocessing functions and comparison
# The goal is to inspect a few realistic processing choices on one MRI image.


def read_image_rgb(image_path):
    """Read an image with OpenCV and convert it to a standard RGB representation."""
    image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_BGRA2RGB)
    else:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image


def read_image_gray(image_path):
    """Read an image and convert it to grayscale."""
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not read grayscale image: {image_path}")
    return image


def resize_and_normalize(gray_image, target_size=(224, 224)):
    """Resize the image to a consistent shape and normalize pixel intensity to [0, 1]."""
    resized = cv2.resize(gray_image, target_size, interpolation=cv2.INTER_AREA)
    normalized = resized.astype(np.float32) / 255.0
    return resized, normalized


def apply_gaussian_blur(gray_image, kernel_size=(5, 5)):
    """Apply Gaussian blur to smooth high-frequency noise."""
    return cv2.GaussianBlur(gray_image, kernel_size, 0)


def apply_otsu_threshold(gray_image):
    """Threshold the grayscale image with Otsu's method; useful only when intensity separation is helpful."""
    _, thresholded = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresholded


def visualize_preprocessing_steps(image_path, class_name):
    """Display original, grayscale, resized, normalized, blurred, and thresholded image views."""
    rgb_image = read_image_rgb(image_path)
    gray_image = cv2.cvtColor(np.array(rgb_image), cv2.COLOR_RGB2GRAY)
    resized_gray, normalized = resize_and_normalize(gray_image)
    blurred = apply_gaussian_blur(gray_image, (5, 5))
    thresholded = apply_otsu_threshold(blurred)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    axes[0].imshow(rgb_image)
    axes[0].set_title(f"Original RGB - {class_name}")
    axes[0].axis('off')

    axes[1].imshow(gray_image, cmap='gray')
    axes[1].set_title("Grayscale")
    axes[1].axis('off')

    axes[2].imshow(resized_gray, cmap='gray')
    axes[2].set_title("Resized to 224x224")
    axes[2].axis('off')

    axes[3].imshow(normalized, cmap='gray')
    axes[3].set_title("Normalized [0, 1]")
    axes[3].axis('off')

    axes[4].imshow(blurred, cmap='gray')
    axes[4].set_title("Gaussian Blur")
    axes[4].axis('off')

    axes[5].imshow(thresholded, cmap='gray')
    axes[5].set_title("Otsu Threshold")
    axes[5].axis('off')

    plt.tight_layout()
    plt.show()


sample_class = 'glioma'
sample_dir = TRAIN_DIR / sample_class
sample_image = list_image_files(sample_dir)[0]
visualize_preprocessing_steps(sample_image, sample_class)

print(f"Sample path: {sample_image}")
print(f"Original image shape: {cv2.imread(str(sample_image), cv2.IMREAD_UNCHANGED).shape}")
print(f"Normalized image min/max: {np.min(cv2.resize(cv2.imread(str(sample_image), cv2.IMREAD_GRAYSCALE), (224, 224), interpolation=cv2.INTER_AREA).astype(np.float32)/255.0):.4f}, {np.max(cv2.resize(cv2.imread(str(sample_image), cv2.IMREAD_GRAYSCALE), (224, 224), interpolation=cv2.INTER_AREA).astype(np.float32)/255.0):.4f}")

# Interpretation:
# Not every MRI image benefits from grayscale conversion or thresholding. We compare the transformed versions visually before choosing a final workflow.
# For some datasets, blur reduces noise but also removes useful detail. Thresholding can help separate the bright tissue from the background only when the intensity distribution supports it.


Sample path: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\data\train\glioma\brisc2025_train_00001_gl_ax_t1.jpg
Original image shape: (512, 512, 3)
Normalized image min/max: 0.0000, 0.9490


# Section 5: Structural and Morphometric Feature Extraction

Hand-crafted features can be useful when the dataset is small or when the pattern is easier to describe numerically than via raw pixels. In this section we compute several simple but defensible image descriptors for each MRI slice.

## Intensity features
These describe the distribution of pixel values in the image:
- mean intensity,
- standard deviation,
- minimum and maximum,
- percentiles such as 10th, 50th, and 90th,
- histogram-based statistics.

These help summarize the brightness and contrast of a scan. Tumor and non-tumor images may differ in intensity distribution even when shapes are similar.

## Shape and region features
Shape descriptors include:
- foreground area ratio,
- bounding box dimensions,
- aspect ratio,
- contour area,
- perimeter,
- circularity.

These can help distinguish regions with different spatial profiles. We must be careful: a simple thresholded mask is not automatically a tumor mask. We treat these as general image descriptors, not medical truth labels.

## Texture and edge features
Texture features summarize local variations in intensity. We use gradient and edge statistics to capture roughness and contrast. These can reflect spatial complexity, which may differ between lesion-bearing and healthy scans.

The important idea is to compare a simple baseline against a richer engineered-feature set and keep only the features that actually help validation performance.


In [10]:
# Hand-crafted feature extraction for MRI slices
# This function intentionally keeps the logic readable and modular.


def safe_float(value):
    return float(value) if np.isfinite(value) else np.nan


def compute_intensity_features(gray_image):
    flat = gray_image.astype(np.float32).ravel()
    hist, _ = np.histogram(flat, bins=256, range=(0, 256))
    hist = hist.astype(np.float32)
    total = hist.sum() + 1e-8
    mean_intensity = float(np.mean(flat))
    std_intensity = float(np.std(flat))
    p10 = float(np.percentile(flat, 10))
    p25 = float(np.percentile(flat, 25))
    p50 = float(np.percentile(flat, 50))
    p75 = float(np.percentile(flat, 75))
    p90 = float(np.percentile(flat, 90))
    min_intensity = float(np.min(flat))
    max_intensity = float(np.max(flat))

    # Histogram-based statistics help summarize brightness and contrast.
    features = {
        'intensity_mean': mean_intensity,
        'intensity_std': std_intensity,
        'intensity_min': min_intensity,
        'intensity_max': max_intensity,
        'intensity_p10': p10,
        'intensity_p25': p25,
        'intensity_p50': p50,
        'intensity_p75': p75,
        'intensity_p90': p90,
        'histogram_mean': float(np.sum(hist * np.arange(256)) / total),
        'histogram_std': float(np.std(hist)),
        'histogram_entropy': float(-np.sum((hist / total) * np.log2(hist / total + 1e-8))),
    }
    return features


def compute_shape_features(gray_image):
    """Create a threshold-mask-based set of geometric descriptors.

    Important: these are image descriptors, not medical tumor masks. They summarize the structure of the image,
    not a validated segmentation of the tumor region.
    """
    blur = cv2.GaussianBlur(gray_image, (5, 5), 0)
    _, thresholded = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    num_pixels = thresholded.shape[0] * thresholded.shape[1]
    foreground_pixels = np.count_nonzero(thresholded > 0)
    foreground_ratio = foreground_pixels / max(num_pixels, 1)

    contours, _ = cv2.findContours(thresholded, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        area = float(cv2.contourArea(largest))
        perimeter = float(cv2.arcLength(largest, True))
        x, y, w, h = cv2.boundingRect(largest)
        aspect_ratio = float(w / max(h, 1))
        circularity = float((4 * np.pi * area) / (perimeter ** 2 + 1e-8))
        bbox_area = float(w * h)
        bbox_fill_ratio = float(area / max(bbox_area, 1))
    else:
        area = 0.0
        perimeter = 0.0
        w = 0.0
        h = 0.0
        aspect_ratio = 0.0
        circularity = 0.0
        bbox_area = 0.0
        bbox_fill_ratio = 0.0

    return {
        'foreground_ratio': float(foreground_ratio),
        'contour_area': float(area),
        'contour_perimeter': float(perimeter),
        'bbox_width': float(w),
        'bbox_height': float(h),
        'bbox_aspect_ratio': float(aspect_ratio),
        'circularity': float(circularity),
        'bbox_fill_ratio': float(bbox_fill_ratio),
    }


def compute_edge_features(gray_image):
    """Estimate edge density and gradient statistics to summarize texture and structure."""
    edges = cv2.Canny(gray_image, 50, 150)
    edge_density = float(np.mean(edges > 0))

    grad_x = cv2.Sobel(gray_image, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray_image, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.hypot(grad_x, grad_y)

    return {
        'edge_density': edge_density,
        'gradient_mean': float(np.mean(grad_mag)),
        'gradient_std': float(np.std(grad_mag)),
        'gradient_max': float(np.max(grad_mag)),
    }


def compute_texture_features(gray_image):
    """A lightweight texture summary based on local variation and gradient energy."""
    local_var = cv2.Laplacian(gray_image, cv2.CV_32F)
    variance = float(np.var(local_var))
    smoothness = float(np.var(gray_image.astype(np.float32)))
    return {
        'texture_variance': variance,
        'texture_smoothness': smoothness,
        'texture_energy': float(np.mean(local_var ** 2)),
    }


def extract_image_features(image_path):
    """Create one feature dictionary from a single MRI image."""
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    resized = cv2.resize(image, (224, 224), interpolation=cv2.INTER_AREA)
    features = {}
    features.update(compute_intensity_features(resized))
    features.update(compute_shape_features(resized))
    features.update(compute_edge_features(resized))
    features.update(compute_texture_features(resized))
    
    # (Optional) If scikit-image is installed, a GLCM texture descriptor can be added here.
    return {key: safe_float(value) for key, value in features.items()}


sample_feature_dict = extract_image_features(sample_image)
print('Feature count:', len(sample_feature_dict))
print('Example features:')
for key, value in list(sample_feature_dict.items())[:10]:
    print(f'  {key}: {value:.4f}')

# Interpretation:
# A single MRI image now has a dense numerical signature. This is easier for classical machine-learning models to use than raw pixel arrays.
# We still must test whether these handcrafted features actually improve validation performance.


Feature count: 27
Example features:
  intensity_mean: 39.2753
  intensity_std: 47.0797
  intensity_min: 0.0000
  intensity_max: 242.0000
  intensity_p10: 0.0000
  intensity_p25: 0.0000
  intensity_p50: 7.0000
  intensity_p75: 75.0000
  intensity_p90: 92.0000
  histogram_mean: 39.2753


# Section 6: Feature Dataset Creation

We now assemble the engineered feature matrix. Each row represents one image, and each column is a structured feature. This creates a dataset that can be fed to classical machine-learning models.

The design choices matter:
- use only the training set to tune hyperparameters,
- do not include file paths or image IDs as model input,
- replace invalid numeric values such as NaN and inf,
- save the engineered dataset for reproducibility.

This step is the bridge between image processing and model training.


In [11]:
# Create feature datasets for train and test partitions.
# This code is deterministic and saves the engineered feature tables for later reuse.


LABELS = ['glioma', 'meningioma', 'pituitary', 'no_tumor']
LABEL_TO_INDEX = {label: idx for idx, label in enumerate(LABELS)}


def build_feature_dataframe(split_dir, split_name):
    """Loop over every image in a split, extract engineered features, and return a DataFrame."""
    rows = []
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        if class_name not in LABEL_TO_INDEX:
            continue
        for image_path in sorted(class_dir.iterdir()):
            if not image_path.is_file():
                continue
            if image_path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}:
                continue
            try:
                features = extract_image_features(image_path)
            except Exception as exc:
                print(f"Skipping unreadable image: {image_path} ({exc})")
                continue

            row = {
                'split': split_name,
                'image_id': image_path.name,
                'class_name': class_name,
                'label': LABEL_TO_INDEX[class_name],
            }
            row.update(features)
            rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No valid image features were produced for {split_name}.")

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(axis=1, how='all')
    df = df.fillna(df.median(numeric_only=True))
    return df


train_feature_df = build_feature_dataframe(TRAIN_DIR, 'train')
test_feature_df = build_feature_dataframe(TEST_DIR, 'test')

print('Training feature shape:', train_feature_df.shape)
print('Testing feature shape:', test_feature_df.shape)
print('Column names (first 10):', list(train_feature_df.columns[:10]))

# Save the engineered feature tables.
train_output_path = OUTPUT_DIR / 'train_engineered_features.csv'
test_output_path = OUTPUT_DIR / 'test_engineered_features.csv'
train_feature_df.to_csv(train_output_path, index=False)
test_feature_df.to_csv(test_output_path, index=False)
print('Saved training features to:', train_output_path)
print('Saved testing features to:', test_output_path)

# Inspect class imbalance.
class_balance = train_feature_df['class_name'].value_counts().sort_index()
print(class_balance)

plt.figure(figsize=(8, 5))
class_balance.plot(kind='bar', color='steelblue')
plt.title('Class balance in the engineered training dataset')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Interpretation:
# The feature dataset is now a structured table that can be used by classical ML models.
# We must keep the feature columns numeric and exclude non-feature identifiers from the model input.


Training feature shape: (5000, 31)
Testing feature shape: (1000, 31)
Column names (first 10): ['split', 'image_id', 'class_name', 'label', 'intensity_mean', 'intensity_std', 'intensity_min', 'intensity_max', 'intensity_p10', 'intensity_p25']
Saved training features to: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\outputs\train_engineered_features.csv
Saved testing features to: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\outputs\test_engineered_features.csv
class_name
glioma        1147
meningioma    1329
no_tumor      1067
pituitary     1457
Name: count, dtype: int64


# Section 7 and 8: Traditional Machine Learning and Evaluation

In this stage we compare classical models using the engineered MRI features. We begin with a baseline, then test more expressive models.

The models we compare include:
- DummyClassifier as a baseline,
- Logistic Regression,
- Support Vector Machine,
- Random Forest,
- Gradient Boosting (if appropriate),
- other simple models if needed.

For each model, we fit only on the training split, use validation data for selection, and only evaluate on the test set after the final configuration is chosen.

We use common classification metrics:
- accuracy,
- precision,
- recall,
- F1-score,
- macro F1,
- confusion matrix.

This helps reveal whether a model is biased toward majority classes or making systematic mistakes between similar tumors.


In [12]:
# Train and compare classical machine-learning models on the engineered feature set.
# We keep the pipeline simple and transparent so beginner learners can follow the logic.

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Use non-feature identifiers and labels separately.
feature_columns = [col for col in train_feature_df.columns if col not in {'split', 'image_id', 'class_name', 'label'}]
X = train_feature_df[feature_columns]
y = train_feature_df['class_name']

# Keep a validation split from the training data. The test data remains untouched until the final model is selected.
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

# Create a few model definitions.
# For models that rely on distance or linear separation, scaling helps.
model_specs = {
    'DummyClassifier': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', DummyClassifier(strategy='most_frequent'))
    ]),
    'LogisticRegression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=2000, multi_class='auto', class_weight='balanced', random_state=SEED))
    ]),
    'SVM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced'))
    ]),
    'RandomForest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', RandomForestClassifier(n_estimators=300, random_state=SEED, class_weight='balanced'))
    ]),
    'GradientBoosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', GradientBoostingClassifier(random_state=SEED))
    ]),
}

results = []
for model_name, model in model_specs.items():
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, val_pred)
    macro_f1 = f1_score(y_val, val_pred, average='macro')
    weighted_f1 = f1_score(y_val, val_pred, average='weighted')
    results.append({
        'model': model_name,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
    })

results_df = pd.DataFrame(results).sort_values(['macro_f1', 'accuracy'], ascending=False)
print(results_df)

# Select the best-performing model on validation data.
selected_model_name = results_df.iloc[0]['model']
selected_model = model_specs[selected_model_name]
selected_model.fit(X, y)

# Now evaluate the final selected model on the held-out test set.
X_test = test_feature_df[feature_columns]
y_test = test_feature_df['class_name']

# Use the exact same preprocessing and feature columns as training.
final_predictions = selected_model.predict(X_test)
final_accuracy = accuracy_score(y_test, final_predictions)
final_macro_f1 = f1_score(y_test, final_predictions, average='macro')
final_weighted_f1 = f1_score(y_test, final_predictions, average='weighted')

print(f'Best validation model: {selected_model_name}')
print(f'Test accuracy: {final_accuracy:.4f}')
print(f'Test macro F1: {final_macro_f1:.4f}')
print(f'Test weighted F1: {final_weighted_f1:.4f}')
print('Classification report:\n', classification_report(y_test, final_predictions, target_names=LABELS, digits=4))

cm = confusion_matrix(y_test, final_predictions, labels=LABELS)
plt.figure(figsize=(8, 6))
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS)
ax.set_title('Confusion matrix on held-out test set')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
plt.tight_layout()
plt.show()

# Interpretation:
# A model can achieve high accuracy on a class-balanced dataset and still perform poorly on minority classes.
# Macro F1 is often more informative for imbalanced classification problems.


                model  accuracy  macro_f1  weighted_f1
3        RandomForest     0.895  0.893029     0.893133
4    GradientBoosting     0.854  0.852089     0.851529
2                 SVM     0.846  0.844927     0.843393
1  LogisticRegression     0.771  0.770876     0.769927
0     DummyClassifier     0.292  0.113003     0.131988
Best validation model: RandomForest
Test accuracy: 0.9170
Test macro F1: 0.9195
Test weighted F1: 0.9153
Classification report:
               precision    recall  f1-score   support

      glioma     0.9512    0.7677    0.8497       254
  meningioma     0.8521    0.9412    0.8944       306
   pituitary     0.9452    0.9857    0.9650       140
    no_tumor     0.9518    0.9867    0.9689       300

    accuracy                         0.9170      1000
   macro avg     0.9251    0.9203    0.9195      1000
weighted avg     0.9202    0.9170    0.9153      1000



# Section 9: Error Analysis and Interpretability

We now inspect the mistakes made by the selected model. This is essential in any educational research project because it reveals which classes are confused and whether the feature design is capturing the right signal.

We ask questions such as:
- Which class pairs are most often confused?
- Are mistakes concentrated in an easy-to-confuse pair such as meningioma and pituitary?
- Are some images visually noisy or poorly centered?
- Are there dataset biases or class imbalance issues?

For interpretability, we can examine feature importances if a tree-based model is selected, or use permutation importance on a validation set. Like all model explanations, these are approximate and should be interpreted carefully.


In [13]:
# Inspect the misclassified examples and explain the pattern.
# This cell is intentionally educational and readable.

from sklearn.inspection import permutation_importance

# Compute confusion statistics to see which class pairs are being confused.
true_labels = y_test.tolist()
pred_labels = final_predictions.tolist()
conf_matrix = confusion_matrix(true_labels, pred_labels, labels=LABELS)
confusion_rows = []
for i, true_label in enumerate(LABELS):
    for j, pred_label in enumerate(LABELS):
        if true_label != pred_label and conf_matrix[i, j] > 0:
            confusion_rows.append({
                'true': true_label,
                'predicted': pred_label,
                'count': conf_matrix[i, j]
            })

print('Confusion matrix:')
print(pd.DataFrame(conf_matrix, index=LABELS, columns=LABELS))

if confusion_rows:
    confusion_df = pd.DataFrame(confusion_rows).sort_values('count', ascending=False)
    print('\nMost common confusion pairs:')
    print(confusion_df.to_string(index=False))
else:
    print('No confusion pairs were found on the test set.')

misclassified_mask = np.array(true_labels) != np.array(pred_labels)
misclassified_indices = np.where(misclassified_mask)[0]
print('\nMisclassified sample count:', len(misclassified_indices))

if len(misclassified_indices) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for axis, idx in zip(axes.flat, misclassified_indices[:6]):
        row = test_feature_df.iloc[idx]
        class_name = row['class_name']
        image_name = row['image_id']
        image_path = TEST_DIR / class_name / image_name
        image_rgb = read_image_rgb(image_path)
        true_label = class_name
        pred_label = final_predictions[idx]
        axis.imshow(image_rgb)
        axis.set_title(f'{true_label} -> {pred_label}')
        axis.axis('off')
    plt.tight_layout()
    plt.show()

# Permutation importance on the validation data helps explain which features matter most.
try:
    perm = permutation_importance(
        selected_model,
        X_val,
        y_val,
        n_repeats=10,
        random_state=SEED,
        scoring='f1_macro'
    )
    importance_df = pd.DataFrame({
        'feature': feature_columns,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    }).sort_values('importance_mean', ascending=False)
    print('\nTop permutation-importance features:')
    print(importance_df.head(10).to_string(index=False))
except Exception as exc:
    print('Permutation importance could not be computed for this model:', type(exc).__name__, exc)

# Interpretation:
# Feature importance is useful but not a medical explanation. At best it tells us which engineered features are
# most useful in this dataset; it does not prove that a specific tissue pattern is clinically decisive.


Confusion matrix:
            glioma  meningioma  pituitary  no_tumor
glioma         195          47         12         0
meningioma       7         288          3         8
pituitary        2           2        296         0
no_tumor         1           1          0       138

Most common confusion pairs:
      true  predicted  count
    glioma meningioma     47
    glioma  pituitary     12
meningioma   no_tumor      8
meningioma     glioma      7
meningioma  pituitary      3
 pituitary     glioma      2
 pituitary meningioma      2
  no_tumor     glioma      1
  no_tumor meningioma      1

Misclassified sample count: 83

Top permutation-importance features:
          feature  importance_mean  importance_std
    intensity_p25         0.029638        0.004997
    histogram_std         0.012108        0.001550
     edge_density         0.005635        0.001306
  bbox_fill_ratio         0.001815        0.000745
 foreground_ratio         0.000974        0.000000
bbox_aspect_ratio         

# Section 10: Deep-Learning Extension

After the engineered-feature baseline is complete, we can compare it with a small CNN built directly on image pixels. A CNN learns its own features automatically by applying filters across the image.

## Key ideas
- Convolution: a small filter slides over the image and computes local dot products.
- Activation: ReLU adds non-linearity.
- Pooling: reduces the spatial size of feature maps and retains the most important information.
- Dense layers: combine the learned features for final class prediction.

A CNN may work better when the dataset is large enough and the images are well aligned. However, classical feature engineering remains useful because it is easier to interpret and often requires less data.

This extension is optional, and it will only run if TensorFlow is installed in the environment.


In [14]:
# CNN baseline: train on the training split, select by validation loss, and evaluate once on held-out test data.

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    print('TensorFlow version:', tf.__version__)
except ImportError:
    tf = None
    print('TensorFlow is unavailable in this notebook kernel. Select the project .venv kernel and rerun.')

if tf is not None:
    img_size = (128, 128)
    batch_size = 32
    class_names = LABELS
    tf.keras.utils.set_random_seed(SEED)
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0, rotation_range=15, width_shift_range=0.10,
        height_shift_range=0.10, horizontal_flip=True, zoom_range=0.10,
        fill_mode='nearest', validation_split=0.2,
    )
    evaluation_datagen = ImageDataGenerator(rescale=1.0 / 255.0, validation_split=0.2)
    train_generator = train_datagen.flow_from_directory(
        str(TRAIN_DIR), classes=class_names, target_size=img_size,
        batch_size=batch_size, class_mode='categorical', subset='training',
        shuffle=True, seed=SEED,
    )
    validation_generator = evaluation_datagen.flow_from_directory(
        str(TRAIN_DIR), classes=class_names, target_size=img_size,
        batch_size=batch_size, class_mode='categorical', subset='validation',
        shuffle=False, seed=SEED,
    )
    cnn_test_generator = ImageDataGenerator(rescale=1.0 / 255.0).flow_from_directory(
        str(TEST_DIR), classes=class_names, target_size=img_size,
        batch_size=batch_size, class_mode='categorical', shuffle=False,
    )

    model = models.Sequential([
        layers.Input(shape=(img_size[0], img_size[1], 3)),
        layers.Conv2D(16, 3, activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'), layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'), layers.Dropout(0.3),
        layers.Dense(len(class_names), activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(
        train_generator, validation_data=validation_generator, epochs=3,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        verbose=1,
    )

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train loss')
    plt.plot(history.history['val_loss'], label='Validation loss')
    plt.title('CNN loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation accuracy')
    plt.title('CNN accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
    plt.tight_layout(); plt.show()

    cnn_probabilities = model.predict(cnn_test_generator, verbose=1)
    cnn_test_predictions = np.argmax(cnn_probabilities, axis=1)
    cnn_test_labels = cnn_test_generator.classes
    cnn_test_accuracy = accuracy_score(cnn_test_labels, cnn_test_predictions)
    cnn_test_macro_f1 = f1_score(cnn_test_labels, cnn_test_predictions, average='macro')
    cnn_test_weighted_f1 = f1_score(cnn_test_labels, cnn_test_predictions, average='weighted')
    comparison_df = pd.DataFrame([
        {'model': f'Engineered features ({selected_model_name})',
         'test_accuracy': final_accuracy, 'test_macro_f1': final_macro_f1,
         'test_weighted_f1': final_weighted_f1},
        {'model': 'CNN', 'test_accuracy': cnn_test_accuracy,
         'test_macro_f1': cnn_test_macro_f1, 'test_weighted_f1': cnn_test_weighted_f1},
    ]).sort_values('test_macro_f1', ascending=False)
    print('Held-out test-set comparison:')
    display(comparison_df)
    print('\nCNN classification report:')
    print(classification_report(
        cnn_test_labels, cnn_test_predictions, labels=list(range(len(class_names))),
        target_names=class_names, digits=4, zero_division=0,
    ))
    cnn_confusion = confusion_matrix(
        cnn_test_labels, cnn_test_predictions, labels=list(range(len(class_names)))
    )
    plt.figure(figsize=(7, 6))
    sns.heatmap(cnn_confusion, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('CNN confusion matrix on held-out test set')
    plt.xlabel('Predicted label'); plt.ylabel('True label'); plt.tight_layout(); plt.show()
    cnn_model_path = OUTPUT_DIR / 'cnn_baseline.keras'
    model.save(cnn_model_path)
    print('Saved CNN model to:', cnn_model_path)
    print('These are educational dataset results, not clinical performance estimates.')
else:
    print('CNN evaluation skipped. Select the project .venv kernel, then rerun this cell.')

TensorFlow version: 2.21.0
Found 4002 images belonging to 4 classes.
Found 998 images belonging to 4 classes.
Found 1000 images belonging to 4 classes.
Epoch 1/3
126/126 ━━━━━━━━━━━━━━━━━━━━ 26s 189ms/step - accuracy: 0.3468 - loss: 1.3314 - val_accuracy: 0.5150 - val_loss: 1.2420
Epoch 2/3
126/126 ━━━━━━━━━━━━━━━━━━━━ 23s 186ms/step - accuracy: 0.4793 - loss: 1.1799 - val_accuracy: 0.4359 - val_loss: 1.2827
Epoch 3/3
126/126 ━━━━━━━━━━━━━━━━━━━━ 23s 185ms/step - accuracy: 0.5580 - loss: 1.0840 - val_accuracy: 0.5371 - val_loss: 1.1732
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step
Held-out test-set comparison:


,model,test_accuracy,test_macro_f1,test_weighted_f1
0,Engineered features (RandomForest),0.917,0.919505,0.915282
1,CNN,0.551,0.571328,0.540039



CNN classification report:
              precision    recall  f1-score   support

      glioma     0.5950    0.4685    0.5242       254
  meningioma     0.3602    0.3072    0.3316       306
   pituitary     0.5764    0.7667    0.6581       300
    no_tumor     0.7714    0.7714    0.7714       140

    accuracy                         0.5510      1000
   macro avg     0.5758    0.5784    0.5713      1000
weighted avg     0.5423    0.5510    0.5400      1000

Saved CNN model to: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\outputs\cnn_baseline.keras
These are educational dataset results, not clinical performance estimates.


In [ ]:
# Purin CNN with channel-wise learned modulation and bounded STP.
# This is an experimental research model; the reported metrics are computed when this cell runs.

import sys
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score
)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
from purin_layers import PurinConv2D, STPModulation

PURIN_IMAGE_SIZE = (224, 224)
PURIN_BATCH_SIZE = 32
PURIN_EPOCHS = 5
PURIN_CLASS_NAMES = list(LABELS)
tf.keras.utils.set_random_seed(SEED)

purin_train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True,
    zoom_range=0.05,
    fill_mode='nearest',
    validation_split=0.2,
)
purin_eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0, validation_split=0.2)
purin_train_generator = purin_train_datagen.flow_from_directory(
    str(TRAIN_DIR), classes=PURIN_CLASS_NAMES, target_size=PURIN_IMAGE_SIZE,
    color_mode='grayscale', batch_size=PURIN_BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True, seed=SEED,
)
purin_validation_generator = purin_eval_datagen.flow_from_directory(
    str(TRAIN_DIR), classes=PURIN_CLASS_NAMES, target_size=PURIN_IMAGE_SIZE,
    color_mode='grayscale', batch_size=PURIN_BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False, seed=SEED,
)

purin_inputs = layers.Input(shape=(224, 224, 1), name='grayscale_mri')
purin_features = purin_inputs
for block_index, filters in enumerate((32, 64, 128, 256, 512), start=1):
    purin_features = PurinConv2D(filters, name=f'purin_conv_{block_index}')(purin_features)
    purin_features = STPModulation(name=f'stp_modulation_{block_index}')(purin_features)
    purin_features = layers.MaxPooling2D(pool_size=2, name=f'pool_{block_index}')(purin_features)
purin_features = layers.Flatten(name='feature_flatten')(purin_features)
purin_features = layers.Dense(465, activation='relu', name='classifier_dense')(purin_features)
purin_features = layers.Dropout(0.35, name='classifier_dropout')(purin_features)
purin_outputs = layers.Dense(len(PURIN_CLASS_NAMES), activation='softmax', name='class_probabilities')(purin_features)
purin_model = tf.keras.Model(purin_inputs, purin_outputs, name='purin_stp_cnn')

adamw_type = getattr(tf.keras.optimizers, 'AdamW', None)
if adamw_type is None:
    adamw_type = tf.keras.optimizers.experimental.AdamW
purin_model.compile(
    optimizer=adamw_type(learning_rate=1e-3, weight_decay=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
print('Purin/STP model parameters:', f'{purin_model.count_params():,}')
print('Input:', PURIN_IMAGE_SIZE, 'grayscale | batch:', PURIN_BATCH_SIZE, '| epochs:', PURIN_EPOCHS)

purin_history = purin_model.fit(
    purin_train_generator,
    validation_data=purin_validation_generator,
    epochs=PURIN_EPOCHS,
    callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
    verbose=1,
)

purin_validation_generator.reset()
purin_validation_loss, _ = purin_model.evaluate(purin_validation_generator, verbose=0)
purin_validation_probabilities = purin_model.predict(purin_validation_generator, verbose=1)
purin_validation_predictions = np.argmax(purin_validation_probabilities, axis=1)
purin_validation_labels = purin_validation_generator.classes
purin_validation_metrics = {
    'accuracy': accuracy_score(purin_validation_labels, purin_validation_predictions),
    'macro_precision': precision_score(
        purin_validation_labels, purin_validation_predictions, average='macro', zero_division=0
    ),
    'macro_recall': recall_score(
        purin_validation_labels, purin_validation_predictions, average='macro', zero_division=0
    ),
    'macro_f1': f1_score(
        purin_validation_labels, purin_validation_predictions, average='macro', zero_division=0
    ),
    'macro_roc_auc': roc_auc_score(
        purin_validation_generator.labels, purin_validation_probabilities,
        labels=list(range(len(PURIN_CLASS_NAMES))), multi_class='ovr', average='macro',
    ),
    'validation_loss': purin_validation_loss,
}
print('\nPurin/STP validation metrics:')
for metric_name, metric_value in purin_validation_metrics.items():
    print(f'{metric_name}: {metric_value:.4f}')
print('\nPer-class validation report:')
print(classification_report(
    purin_validation_labels, purin_validation_predictions,
    labels=list(range(len(PURIN_CLASS_NAMES))), target_names=PURIN_CLASS_NAMES,
    digits=4, zero_division=0,
))

purin_model_path = OUTPUT_DIR / 'purin_stp_cnn.keras'
purin_model.save(purin_model_path)
print('Saved Purin/STP model to:', purin_model_path)

# Grad-CAM uses the final Purin convolution map; it is a model explanation, not clinical evidence.
gradcam_model = tf.keras.Model(
    purin_model.inputs,
    [purin_model.get_layer('purin_conv_5').output, purin_model.output],
)
purin_validation_generator.reset()
purin_images, purin_image_labels = purin_validation_generator[0]
purin_image_batch = purin_images[:1]
with tf.GradientTape() as grad_tape:
    final_conv_maps, predicted_probabilities = gradcam_model(purin_image_batch, training=False)
    explained_class = tf.argmax(predicted_probabilities[0])
    class_score = predicted_probabilities[:, explained_class]
map_gradients = grad_tape.gradient(class_score, final_conv_maps)
channel_importance = tf.reduce_mean(map_gradients, axis=(1, 2), keepdims=True)
gradcam_heatmap = tf.reduce_sum(channel_importance * final_conv_maps, axis=-1)[0]
gradcam_heatmap = tf.maximum(gradcam_heatmap, 0)
gradcam_heatmap /= tf.reduce_max(gradcam_heatmap) + tf.keras.backend.epsilon()
gradcam_heatmap = tf.image.resize(
    gradcam_heatmap[None, ..., None], PURIN_IMAGE_SIZE, method='bilinear'
)[0, ..., 0].numpy()

plt.figure(figsize=(7, 6))
plt.imshow(purin_image_batch[0, ..., 0], cmap='gray')
plt.imshow(gradcam_heatmap, cmap='jet', alpha=0.4)
plt.title(
    f"Grad-CAM | predicted: {PURIN_CLASS_NAMES[int(explained_class)]} "
    f"| actual: {PURIN_CLASS_NAMES[int(purin_image_labels[0].argmax())]}"
)
plt.axis('off')
plt.tight_layout()
plt.show()


Found 4002 images belonging to 4 classes.
Found 998 images belonging to 4 classes.

Purin/STP model parameters: 13,238,233
Input: (224, 224) grayscale | batch: 32 | epochs: 5
Epoch 1/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step - accuracy: 0.5925 - loss: 0.9387 - val_accuracy: 0.6974 - val_loss: 2.3561
Epoch 2/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - accuracy: 0.7444 - loss: 0.6300 - val_accuracy: 0.7265 - val_loss: 2.2132
Epoch 3/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 145s 1s/step - accuracy: 0.7966 - loss: 0.5187 - val_accuracy: 0.7956 - val_loss: 1.5600
Epoch 4/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 146s 1s/step - accuracy: 0.8051 - loss: 0.4652 - val_accuracy: 0.7325 - val_loss: 2.7696
Epoch 5/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - accuracy: 0.8358 - loss: 0.4064 - val_accuracy: 0.8226 - val_loss: 1.2080
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 299ms/step

Purin/STP validation metrics:
accuracy: 0.8226
macro_precision: 0.8305
macro_recall: 0.8307
macro_f1: 0.8093
macro_roc_auc: 0.9301
validation_

# Section 11: Saving and Reusing the Model

Once the final model is selected, we save the trained model and make sure the same preprocessing and feature extraction workflow is used during inference. This turns the notebook into a reusable research pipeline.

We save:
- the trained classifier,
- the feature column list,
- the class label dictionary,
- the model metadata needed for future predictions.


In [19]:
# Save or restore the classical and Purin/STP models plus their inference metadata.

import json
import sys
from pathlib import Path
import cv2
import joblib
import numpy as np
import tensorflow as tf

OUTPUT_DIR = Path(globals().get('OUTPUT_DIR', Path.cwd() / 'outputs'))
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
project_dir = Path(globals().get('WORKSPACE_ROOT', Path.cwd()))
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
from purin_layers import PurinConv2D, STPModulation

classifier_path = OUTPUT_DIR / 'selected_classifier.joblib'
feature_columns_path = OUTPUT_DIR / 'feature_columns.joblib'
label_names_path = OUTPUT_DIR / 'label_names.joblib'
purin_model_path = OUTPUT_DIR / 'purin_stp_cnn.keras'
purin_metrics_path = OUTPUT_DIR / 'purin_stp_metrics.json'

if 'selected_model' in globals():
    joblib.dump(selected_model, classifier_path)
elif classifier_path.exists():
    selected_model = joblib.load(classifier_path)
else:
    raise FileNotFoundError('Run the classical model training cell or provide its saved classifier.')

if 'feature_columns' in globals():
    joblib.dump(feature_columns, feature_columns_path)
elif feature_columns_path.exists():
    feature_columns = joblib.load(feature_columns_path)
else:
    raise FileNotFoundError('Run the feature extraction cell or provide saved feature columns.')

if 'LABELS' in globals():
    class_labels = list(LABELS)
elif label_names_path.exists():
    class_labels = joblib.load(label_names_path)
else:
    raise FileNotFoundError('Class labels are unavailable; run the dataset setup cell first.')
joblib.dump(class_labels, label_names_path)
LABELS = class_labels

if 'purin_model' in globals():
    purin_model.save(purin_model_path)
elif not purin_model_path.exists():
    raise FileNotFoundError('Run the Purin/STP training cell or provide outputs/purin_stp_cnn.keras.')

purin_model = tf.keras.models.load_model(
    purin_model_path,
    custom_objects={
        'PurinConv2D': PurinConv2D,
        'STPModulation': STPModulation,
        'PurinMRI>PurinConv2D': PurinConv2D,
        'PurinMRI>STPModulation': STPModulation,
    },
    compile=False,
)

if 'purin_validation_metrics' in globals():
    purin_classification_report = classification_report(
        purin_validation_labels,
        purin_validation_predictions,
        labels=list(range(len(class_labels))),
        target_names=class_labels,
        output_dict=True,
        zero_division=0,
    )
    purin_report_data = {
        'metrics': {name: float(value) for name, value in purin_validation_metrics.items()},
        'class_report': purin_classification_report,
        'class_labels': class_labels,
        'input_size': list(PURIN_IMAGE_SIZE),
        'batch_size': int(PURIN_BATCH_SIZE),
        'epochs_configured': int(PURIN_EPOCHS),
        'epochs_completed': len(purin_history.history.get('loss', [])),
        'parameter_count': int(purin_model.count_params()),
    }
    with purin_metrics_path.open('w', encoding='utf-8') as metrics_file:
        json.dump(purin_report_data, metrics_file, indent=2)
elif purin_metrics_path.exists():
    with purin_metrics_path.open('r', encoding='utf-8') as metrics_file:
        purin_report_data = json.load(metrics_file)
    purin_validation_metrics = purin_report_data['metrics']
    purin_classification_report = purin_report_data['class_report']
else:
    raise FileNotFoundError('Purin/STP evaluation metrics are unavailable; run its evaluation cell first.')

print('Saved classical classifier:', classifier_path)
print('Saved feature columns:', feature_columns_path)
print('Saved class labels:', label_names_path)
print('Saved and reloaded Purin/STP model:', purin_model_path)
print('Saved Purin/STP metrics:', purin_metrics_path)
print('Reloaded model parameter count:', f'{purin_model.count_params():,}')


def predict_single_image(model, image_path):
    """Predict with the engineered-feature classifier."""
    feature_values = extract_image_features(image_path)
    feature_row = pd.DataFrame([feature_values], columns=feature_columns)
    prediction = model.predict(feature_row)[0]
    probabilities = model.predict_proba(feature_row)[0] if hasattr(model, 'predict_proba') else None
    return prediction, probabilities


def predict_purin_image(image_path, model=None):
    """Predict one grayscale MRI image with the saved Purin/STP model."""
    image_gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image_gray is None:
        raise ValueError(f'Could not read image: {image_path}')
    image_gray = cv2.resize(image_gray, tuple(PURIN_IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    image_batch = image_gray.astype(np.float32)[None, ..., None] / 255.0
    probabilities = (model or purin_model).predict(image_batch, verbose=0)[0]
    predicted_label = class_labels[int(np.argmax(probabilities))]
    return predicted_label, probabilities


if 'extract_image_features' in globals() and 'TRAIN_DIR' in globals():
    example_image = next(path for path in (TRAIN_DIR / class_labels[0]).glob('*') if path.is_file())
    classical_prediction, _ = predict_single_image(selected_model, example_image)
    purin_prediction, purin_probabilities = predict_purin_image(example_image)
    print('Example image:', example_image.name)
    print('Engineered-feature prediction:', classical_prediction)
    print('Purin/STP prediction:', purin_prediction)
    print('Purin/STP probabilities:', dict(zip(class_labels, purin_probabilities.round(4))))
else:
    print('Models restored. Run the feature and dataset setup cells to enable sample inference.')


Saved classical classifier: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\outputs\selected_classifier.joblib
Saved feature columns: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\outputs\feature_columns.joblib
Saved class labels: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\outputs\label_names.joblib
Saved and reloaded Purin/STP model: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\outputs\purin_stp_cnn.keras
Saved Purin/STP metrics: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\outputs\purin_stp_metrics.json
Reloaded model parameter count: 13,238,233
Example image: brisc2025_train_00001_gl_ax_t1.jpg
Engineered-feature prediction: glioma
Purin/STP prediction: meningioma
Purin/STP probabilities: {'glioma': np.float32(0.108), 'meningioma': np.float32(0.4328), 'pituitary': np.float32(0.4326), 'no_tumor': np.float32(0.0266)}


# Section 12: Final Report

## Research summary
This project compares engineered image features, a compact CNN baseline, and an experimental Purin/STP convolutional model for four-class brain MRI classification: glioma, meningioma, pituitary tumor, and no tumor.

## Methodology
1. Inspect the dataset and confirm the folder structure.
2. Extract engineered intensity, shape, texture, and edge features.
3. Train and select classical models using a stratified validation split.
4. Train the compact CNN baseline and evaluate it on the held-out test set.
5. Train the Purin/STP experiment on grayscale 224x224 images with batch size 32, AdamW, and five epochs.
6. Evaluate Purin/STP on the 998-image validation subset, report classification metrics, and create a Grad-CAM visualization.

## Existing held-out test results

| Model | Accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|
| Engineered features + Random Forest | 0.9170 | 0.9195 | 0.9153 |
| Compact CNN baseline | 0.5510 | 0.5713 | 0.5400 |

These are held-out test results for the original baselines. The compact CNN is a short three-epoch, 128x128 baseline and is not a tuned deep-learning benchmark.

## Purin/STP validation results

The training folder supplied 4,002 training images and a 998-image validation subset. The Purin/STP model has 13,238,233 parameters and uses five Purin convolution stages with learnable channel-wise output modulation and activation-statistic-driven STP states clipped to [0.8, 1.2], recovering toward 1.0.

| Validation metric | Result |
|---|---:|
| Accuracy | 82.26% |
| Macro precision | 83.05% |
| Macro recall | 83.07% |
| Macro F1 | 80.93% |
| Macro ROC-AUC (OvR) | 93.01% |
| Validation loss | 1.2080 |

| Class | Precision | Recall | F1 | Support |
|---|---:|---:|---:|---:|
| Glioma | 79.06% | 95.63% | 86.56% | 229 |
| Meningioma | 89.93% | 47.17% | 61.88% | 265 |
| Pituitary | 84.83% | 94.16% | 89.25% | 291 |
| No tumor | 78.38% | 95.31% | 86.02% | 213 |

These are validation results, not held-out test results, and should not be compared directly with the baseline test scores above. Meningioma recall is the main weakness in this run.

## Saving and reuse
The trained model is saved as `outputs/purin_stp_cnn.keras`; custom layers are defined in `purin_layers.py` so the model can be loaded after a kernel restart. Section 11 saves the classical model and its feature metadata, class labels, plus Purin/STP metrics and class report in `outputs/purin_stp_metrics.json`. The notebook also exposes `predict_purin_image` for single-image inference.

The PDF report is generated at `outputs/brain_mri_classification_report.pdf` by the next code cell.

## Limitations
- This is an educational project, not a medically validated diagnostic system.
- Dataset-specific performance may reflect image-source artifacts or split characteristics; external clinical performance is unknown.
- Grad-CAM and engineered-feature importance are approximate explanations, not clinical evidence.
- Scores are not calibrated medical confidence estimates.

## Conclusion
The Purin/STP model completed training and reached 82.26% validation accuracy, 80.93% macro F1, and 93.01% macro ROC-AUC in this run. The saved-model reload and single-image inference path were also verified. The results support further experimentation, but the meningioma recall and external validation require attention before making broader performance claims.


In [20]:
# Export the current experiment and baseline findings as a multi-page PDF.

from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(globals().get('OUTPUT_DIR', Path.cwd() / 'outputs'))
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
report_pdf_path = OUTPUT_DIR / 'brain_mri_classification_report.pdf'
report_metrics = purin_report_data['metrics']
report_labels = purin_report_data['class_labels']
report_class_details = purin_report_data['class_report']

with PdfPages(report_pdf_path) as pdf:
    fig, axis = plt.subplots(figsize=(8.27, 11.69))
    axis.axis('off')
    axis.text(0.06, 0.96, 'Brain MRI Classification Report', fontsize=21, weight='bold', va='top')
    axis.text(0.06, 0.915, 'Purin/STP validation run | research use only', fontsize=11, color='#4b5563', va='top')
    axis.text(
        0.06, 0.86,
        f"Input: {purin_report_data['input_size'][0]}x{purin_report_data['input_size'][1]} grayscale | "
        f"Batch: {purin_report_data['batch_size']} | Epochs completed: {purin_report_data['epochs_completed']}\n"
        f"Parameters: {purin_report_data['parameter_count']:,} | Validation samples: 998",
        fontsize=10, va='top', linespacing=1.6,
    )
    axis.text(0.06, 0.78, 'Purin/STP validation metrics', fontsize=14, weight='bold', va='top')
    metric_rows = [
        ['Accuracy', f"{report_metrics['accuracy']:.4f}"],
        ['Macro precision', f"{report_metrics['macro_precision']:.4f}"],
        ['Macro recall', f"{report_metrics['macro_recall']:.4f}"],
        ['Macro F1', f"{report_metrics['macro_f1']:.4f}"],
        ['Macro ROC-AUC (OvR)', f"{report_metrics['macro_roc_auc']:.4f}"],
        ['Validation loss', f"{report_metrics['validation_loss']:.4f}"],
    ]
    axis.table(
        cellText=metric_rows, colLabels=['Metric', 'Result'], cellLoc='left', colLoc='left',
        bbox=[0.06, 0.49, 0.88, 0.25], colWidths=[0.62, 0.26],
    )
    axis.text(0.06, 0.43, 'Existing held-out test baselines', fontsize=14, weight='bold', va='top')
    baseline_rows = [
        ['Engineered features + Random Forest', '0.9170', '0.9195', '0.9153'],
        ['Compact CNN baseline', '0.5510', '0.5713', '0.5400'],
    ]
    axis.table(
        cellText=baseline_rows,
        colLabels=['Model', 'Accuracy', 'Macro F1', 'Weighted F1'],
        cellLoc='left', colLoc='left', bbox=[0.06, 0.27, 0.88, 0.12],
        colWidths=[0.48, 0.16, 0.16, 0.18],
    )
    axis.text(
        0.06, 0.21,
        'Purin/STP figures are from the validation split; baseline figures are from the held-out test set. '
        'These splits differ, so the scores are not a direct model comparison. This educational model is '
        'not validated for clinical use.',
        fontsize=9, va='top', wrap=True, linespacing=1.5,
    )
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    class_rows = []
    class_f1 = []
    for class_label in report_labels:
        class_metrics = report_class_details[class_label]
        class_f1.append(class_metrics['f1-score'])
        class_rows.append([
            class_label.replace('_', ' ').title(),
            f"{class_metrics['precision']:.4f}",
            f"{class_metrics['recall']:.4f}",
            f"{class_metrics['f1-score']:.4f}",
            str(int(class_metrics['support'])),
        ])
    fig, axis = plt.subplots(figsize=(8.27, 11.69))
    axis.axis('off')
    axis.text(0.06, 0.96, 'Class-wise validation results', fontsize=19, weight='bold', va='top')
    axis.table(
        cellText=class_rows,
        colLabels=['Class', 'Precision', 'Recall', 'F1', 'N'],
        cellLoc='left', colLoc='left', bbox=[0.06, 0.75, 0.88, 0.15],
        colWidths=[0.30, 0.18, 0.18, 0.16, 0.10],
    )
    bar_axis = fig.add_axes([0.14, 0.36, 0.76, 0.24])
    bar_axis.barh([label.replace('_', ' ').title() for label in report_labels], class_f1, color='#167c80')
    bar_axis.set_xlim(0, 1)
    bar_axis.set_xlabel('Validation F1')
    bar_axis.grid(axis='x', alpha=0.2)
    bar_axis.invert_yaxis()
    axis.text(
        0.06, 0.27,
        'Meningioma recall is the main weakness in this run. Grad-CAM provides a visual summary of '
        'model sensitivity, not a medical explanation.',
        fontsize=10, va='top', wrap=True, linespacing=1.5,
    )
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    if 'gradcam_heatmap' in globals() and 'purin_image_batch' in globals():
        fig, axis = plt.subplots(figsize=(8.27, 11.69))
        axis.imshow(purin_image_batch[0, ..., 0], cmap='gray')
        axis.imshow(gradcam_heatmap, cmap='jet', alpha=0.42)
        axis.set_title('Grad-CAM example from the validation workflow', fontsize=16, pad=16)
        axis.axis('off')
        fig.text(
            0.08, 0.08,
            'Qualitative model visualization only; it does not establish tumor localization or clinical validity.',
            fontsize=9, ha='left',
        )
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print('Created PDF report:', report_pdf_path)
print('PDF size:', f'{report_pdf_path.stat().st_size:,}', 'bytes')


Created PDF report: c:\Users\KIIT\OneDrive\Desktop\Brain Tumour\outputs\brain_mri_classification_report.pdf
PDF size: 231,786 bytes
